In [19]:
import os
from pathlib import Path

if Path.cwd().name != "book-text-ml":
    for candidate in [Path("book-text-ml"), *Path(".").glob("*/book-text-ml")]:
        if candidate.is_dir():
            os.chdir(candidate)
            break

print("작업 폴더:", Path.cwd())

DATA_PATH = "book_bestseller_clean.csv"


작업 폴더: c:\dev\llm-data-analysis-course\notebooks\book-text-ml


In [20]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [21]:
DATA_PATH = "book_bestseller_clean.csv"
df_books = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig",
)
print("데이터 크기:", df_books.shape)
print("컬럼:", df_books.columns.tolist())
print("상품명 결측치:", df_books["상품명"].isna().sum())
df_books[["상품명"]].head(10)

데이터 크기: (986, 8)
컬럼: ['순위', '판매상품ID', '상품명', '판매가', '저자', '출판사', '발행일', '분야']
상품명 결측치: 0


,상품명
0,"세네카, 오늘을 빼앗기고 있는 당신에게"
1,흔한남매 23
2,머니 트렌드 2027
3,싯다르타
4,한국사 이상현상 연구원(일반판)
5,테오
6,쇼펜하우어 인생수업(30만 부 기념 개정증보판)
7,니체의 초월자
8,판매의 법칙
9,마음의 어휘력


In [22]:

df_reco = df_books.copy()

df_reco["상품명"] = (

    df_reco["상품명"]

    .fillna("")

    .astype(str)

    .str.strip()

)

df_reco = (

    df_reco[df_reco["상품명"] != ""]

    .reset_index(drop=True)

)

print("추천에 사용할 도서 수:", len(df_reco))

추천에 사용할 도서 수: 986


In [23]:
#실습 3. 추천용 데이터 준비하기
df_reco = df_books.copy()
df_reco["상품명"] = (
    df_reco["상품명"]
    .fillna("")
    .astype(str)
    .str.strip()
)
df_reco = (
    df_reco[df_reco["상품명"] != ""]
    .reset_index(drop=True)
)
print("추천에 사용할 도서 수:", len(df_reco))

추천에 사용할 도서 수: 986


실습 4. 콘텐츠 기반 추천 이해하기

이번 실습에서는 콘텐츠 기반 추천(Content-Based Recommendation) 을 사용합니다.

실습 5. 코사인 유사도 이해하기

In [24]:
#실습 6. 실제 도서 제목을 TF-IDF로 변환하기


titles = df_reco["상품명"]
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(titles)
print("도서 수:", tfidf_matrix.shape[0])
print("단어 수:", tfidf_matrix.shape[1])

도서 수: 986
단어 수: 2179


In [25]:
#실습 7. 기준 도서 선택하기
#먼저 한 권을 index로 선택합니다.

# 추천의 기준이 될 도서를 번호(index)로 하나 고른다
selected_index = 0

# 그 번호에 해당하는 도서의 상품명을 가져온다
selected_title = df_reco.loc[
    selected_index,
    "상품명",
]

print("선택 번호:", selected_index)
print("선택 도서:", selected_title)
print("분야     :", df_reco.loc[selected_index, "분야"])

선택 번호: 0
선택 도서: 세네카, 오늘을 빼앗기고 있는 당신에게
분야     : 인문


In [26]:
#실습 8. 선택 도서와 전체 도서의 유사도 계산하기

#선택한 도서의 벡터를 가져옵니다.

# 1) 선택한 도서의 TF-IDF 벡터를 가져온다
selected_vector = tfidf_matrix[selected_index]

print("선택 도서:", selected_title)
print("선택 벡터 크기:", selected_vector.shape)   # (1, 단어 수) 여야 한다

선택 도서: 세네카, 오늘을 빼앗기고 있는 당신에게
선택 벡터 크기: (1, 2179)


In [27]:
# 2) 선택 도서와 전체 도서의 코사인 유사도를 계산한다
from sklearn.metrics.pairwise import cosine_similarity

similarity_scores = cosine_similarity(
    selected_vector,
    tfidf_matrix,
).flatten()   # (1, 도서 수) 모양을 1차원으로 펼친다

print("유사도 개수:", len(similarity_scores))
print("전체 도서 수:", len(df_reco))
print("두 값이 같은가?:", len(similarity_scores) == len(df_reco))

유사도 개수: 986
전체 도서 수: 986
두 값이 같은가?: True


In [28]:
# 3) 유사도 점수와 원본 도서가 순서대로 대응하는지 확인한다
for i in range(3):
    print(f"similarity_scores[{i}] = {similarity_scores[i]:.4f}  <->  {df_reco.iloc[i]['상품명']}")

print()
print("자기 자신과의 유사도:", round(similarity_scores[selected_index], 4))   # 1.0이어야 한다

similarity_scores[0] = 1.0000  <->  세네카, 오늘을 빼앗기고 있는 당신에게
similarity_scores[1] = 0.0000  <->  흔한남매 23
similarity_scores[2] = 0.0000  <->  머니 트렌드 2027

자기 자신과의 유사도: 1.0


In [29]:
###실습 9. 유사도가 높은 순서 확인하기
#먼저 결과를 제목과 함께 확인합니다.

score_df = pd.DataFrame({
    "index": np.arange(len(df_reco)),
    "상품명": df_reco["상품명"],
    "similarity": similarity_scores,
})
score_df.sort_values(
    "similarity",
    ascending=False,
).head(10)

,index,상품명,similarity
0,0,"세네카, 오늘을 빼앗기고 있는 당신에게",1.000000
587,587,"오늘을 빼앗기는 당신에게, 세네카",0.668201
554,554,돌이킬 수 있는,0.252116
659,659,남아 있는 나날,0.203866
521,521,비전공자도 이해할 수 있는 LLM 수업,0.161707
349,349,품격 있는 대화를 위한 지식 브리핑,0.147451
928,928,비전공자도 이해할 수 있는 AI 지식(10만부 기념 개정판),0.132410
4,4,한국사 이상현상 연구원(일반판),0.000000
5,5,테오,0.000000
6,6,쇼펜하우어 인생수업(30만 부 기념 개정증보판),0.000000


실습 10. 자기 자신을 제외하고 Top 5 만들기

유사도가 높은 순서의 index를 구합니다.

### 실습 11. 추천 결과에 메타데이터 추가하기


In [32]:
available_columns = [
    column
    for column in [
        "상품명",
        "인물",
        "출판사",
        "분야",
    ]
    if column in df_reco.columns
]
result = df_reco.loc[
    recommended_indices,
    available_columns,
].copy()
result["similarity"] = [
    round(float(similarity_scores[idx]), 4)
    for idx in recommended_indices
]
result

NameError: name 'recommended_indices' is not defined